In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from category_encoders import LeaveOneOutEncoder
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
import joblib
import optuna  
import os

# Fonction de pipeline pour la préparation des données 
def prepare_data(df_raw):
    df = df_raw.copy()  # Pour éviter de modifier l'original
    
    # Suppression colonnes inutiles
    to_drop = ['id', 'titre', 'url_annonce', 'vendeur', 'Numero_telephone', 'date_scraping', 'date_annonce', 'traitee', 'utile']
    df.drop(columns=[c for c in to_drop if c in df.columns], inplace=True)
    
    # Filtrer seulement les voitures
    df = df[df['description'].str.contains('voiture|car|auto|vehicule', case=False, na=False) | df['source'].str.contains('voitures', case=False, na=False)]
    
    def clean_price(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().lower()
        if 'négociable' in x or x == '':
            return np.nan
        digits = re.findall(r'\d+', x)
        if not digits:
            return np.nan
        val = int(''.join(digits))
        if val < 5000:
            val *= 10
        return val
    
    df['prix'] = df['prix'].apply(clean_price)
    
    def clean_kilometrage(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().lower()
        digits = re.findall(r'\d+', x)
        if not digits:
            return np.nan
        val = int(''.join(digits))
        if val < 1000:
            val *= 1000
        return val
    
    df['kilometrage'] = df['kilometrage'].apply(clean_kilometrage)
    
    def clean_puissance(x):
        if pd.isna(x):
            return np.nan
        match = re.search(r'(\d+\.?\d*)\s*(cv|chevaux)?', str(x), re.IGNORECASE)
        return float(match.group(1)) if match else np.nan
    df['puissance_fiscale'] = df['puissance_fiscale'].apply(clean_puissance)
    
    def clean_year(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip()
        match = re.search(r'(19|20)\d{2}', x)
        if match:
            return int(match.group(0))
        return np.nan
    
    df['mise_en_circulation'] = df['mise_en_circulation'].apply(clean_year)
    
    # Suppression des lignes sans prix
    df.dropna(subset=['prix'], inplace=True)
    
    # Calcul de l'âge
    current_year = 2025
    df['age'] = current_year - df['mise_en_circulation']
    df['age'] = df['age'].clip(lower=1)
    
    # Nouvelle feature: km_per_year
    df['km_per_year'] = df['kilometrage'] / df['age']
    
    # Log transform sur features skew
    df['kilometrage_log'] = np.log1p(df['kilometrage'])
    df['km_per_year_log'] = np.log1p(df['km_per_year'])
    
    # Extraction engine_size depuis description
    def extract_engine_size(desc):
        if pd.isna(desc):
            return np.nan
        desc = str(desc).lower()
        match = re.search(r'(\d[\.,]\d)\s*(l|tsi|cdi|tdi|essence|diesel)', desc)
        if match:
            return float(match.group(1).replace(',', '.'))
        return np.nan
    
    df['engine_size'] = df['description'].apply(extract_engine_size)
    
    # Interaction feature
    df['power_engine_interact'] = df['puissance_fiscale'] * df['engine_size']
    
    # Remplacement missing numériques par médiane (par groupe marque/modele si possible)
    num_cols = ['kilometrage', 'puissance_fiscale', 'age', 'km_per_year', 'engine_size', 'kilometrage_log', 'km_per_year_log', 'power_engine_interact']
    for col in num_cols:
        if df[col].isna().sum() > 0:
            df[col] = df.groupby(['marque', 'modele'])[col].transform(lambda x: x.fillna(x.median()))
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
    
    # Catégorielles
    cat_cols = ['etat', 'carburant', 'boite_de_vitesse', 'marque', 'modele', 'couleur', 'source']
    for col in cat_cols:
        df[col] = df[col].fillna('inconnu')         # Remplacer NaN par 'inconnu' avant conversion
        df[col] = df[col].astype(str).str.lower().str.strip()
        df[col] = df[col].replace({'': 'inconnu'})  # Remplacer chaînes vides après nettoyage


    if 'ville' in df.columns:
        df['ville'] = df['ville'].astype(str).str.lower().str.strip().fillna('inconnu').replace({'': 'inconnu'})
        cat_cols.append('ville')
    
    # Target encoding pour high cardinality avec LeaveOneOutEncoder
    encoder = LeaveOneOutEncoder(sigma=0.1)  # Sigma pour ajouter du bruit et éviter overfit
    encoding_cols = ['marque', 'modele']
    if 'ville' in df.columns:
        encoding_cols.append('ville')
    df[[f'{col}_encoded' for col in encoding_cols]] = encoder.fit_transform(df[encoding_cols], df['prix'])
    
    # Convertir les autres en category pour LGBM
    for col in cat_cols:
        df[col] = df[col].astype('category')
    
    # Extraction features depuis description 
    def extract_features_from_desc(desc):
        if pd.isna(desc):
            return {}
        desc = str(desc).lower()
        features = {
            'premiere_main': 1 if 'première main' in desc else 0,
            'toit_ouvrant': 1 if 'toit ouvrant' in desc or 'panoramique' in desc else 0,
            'camera_recul': 1 if 'caméra' in desc or 'recul' in desc else 0,
            'gps': 1 if 'gps' in desc or 'navigation' in desc else 0,
            'bluetooth': 1 if 'bluetooth' in desc else 0,
            'jantes_alu': 1 if 'jantes alu' in desc else 0,
            'climatronic': 1 if 'climatronic' in desc or 'clim' in desc else 0,
            'sieges_cuir': 1 if 'cuir' in desc and 'siège' in desc else 0,
            'radar': 1 if 'radar' in desc else 0,
            'full_options': 1 if 'full options' in desc or 'toute option' in desc else 0,
            'airbags': 1 if 'airbag' in desc else 0,
            'regulateur_vitesse': 1 if 'régulateur' in desc or 'cruise' in desc else 0,
            'abs_esp': 1 if 'abs' in desc or 'esp' in desc else 0,
            'importee': 1 if 'importée' in desc or 'importee' in desc else 0,
            'nb_portes': 5 if '5 portes' in desc else (3 if '3 portes' in desc else 0),
            'suv': 1 if 'suv' in desc else 0,
            'berline': 1 if 'berline' in desc else 0,
            'coupe': 1 if 'coupé' in desc or 'coupe' in desc else 0,
        }
        return features
    
    extra_feats = df['description'].apply(extract_features_from_desc)
    extra_df = pd.DataFrame(list(extra_feats))
    df = pd.concat([df, extra_df], axis=1)
    
    # Marque premium
    premium_brands = ['bmw', 'mercedes', 'audi', 'porsche', 'lexus', 'jaguar', 'land rover']
    df['is_premium'] = df['marque'].str.lower().isin(premium_brands).astype(int)
    
    # Drop description
    df.drop(columns=['description'], inplace=True)
    
    # Filtrer outliers avec Isolation Forest
    def filter_outliers(df, cols=['prix', 'kilometrage']):
        iso = IsolationForest(contamination=0.05, random_state=42)
        mask = iso.fit_predict(df[cols].dropna()) != -1
        return df[df.index.isin(df[cols].dropna().index[mask])]
    
    df = filter_outliers(df)
    
    # Filtrage manuel supplémentaire
    df = df[(df['prix'] >= 10000) & (df['prix'] <= 300000)]
    df = df[(df['kilometrage'] >= 1000) & (df['kilometrage'] <= 500000)]
    df = df[(df['mise_en_circulation'] >= 1990) & (df['mise_en_circulation'] <= 2025)]
    
    # Supprimer doublons
    df.drop_duplicates(inplace=True)
    
    # Standardisation des variables numériques
    scaler = StandardScaler()
    num_cols_to_scale = ['puissance_fiscale', 'engine_size', 'power_engine_interact']
    df[num_cols_to_scale] = scaler.fit_transform(df[num_cols_to_scale])
    
    # Préparation finale
    X = df.drop('prix', axis=1)
    y = np.log1p(df['prix'])
    
    return X, y, encoder, cat_cols, scaler

# Chargement 
df = pd.read_csv("C:/Users/maram/Downloads/annonces_brutes_202507021126/annonces_brutes_202507021126.csv", sep=';', low_memory=False)

# Application du pipeline de préparation 
X, y, encoder, cat_cols, scaler = prepare_data(df)

# Hyperparameter tuning avec Optuna 
def objective(trial, X, y, cat_cols):
    params = {
        'objective': 'regression',
        'metric': ['mae', 'rmse'],
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 256),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 50),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 10.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 10.0),
        'verbose': -1,
        'seed': 42
    }

    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    rmse_list = []
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols)
        val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, reference=train_data)

        model = lgb.train(
            params,
            train_data,
            valid_sets=[train_data, val_data],
            num_boost_round=5000,
            callbacks=[lgb.early_stopping(200), lgb.log_evaluation(False)]
        )

        y_pred_log = model.predict(X_val, num_iteration=model.best_iteration)
        y_pred = np.expm1(y_pred_log)
        y_val_orig = np.expm1(y_val)
        rmse_list.append(np.sqrt(mean_squared_error(y_val_orig, y_pred)))

    return np.mean(rmse_list)


def build_and_train_pipeline(
    df_path=None, 
    df_raw=None,
    do_optuna=True,
    n_trials=20,
    save_prefix='modele_lightgbm_voiture'
):
    # Charger les données
    if df_raw is None and df_path is not None:
        print(f"[INFO] Chargement des données depuis : {df_path}")
        df_raw = pd.read_csv(df_path, sep=';', low_memory=False)
        print(f"[INFO] Données chargées : {df_raw.shape[0]} lignes, {df_raw.shape[1]} colonnes")
    elif df_raw is None:
        raise ValueError("Fournir soit df_raw soit df_path")

    # Préparation des données
    print("[INFO] Préparation des données...")
    X, y, encoder, cat_cols, scaler = prepare_data(df_raw)
    print(f"[INFO] Données préparées : X={X.shape}, y={y.shape}")
    print("[INFO] Exemple de données préparées (X):")
    display(X.head())

    # Optuna tuning
    if do_optuna:
        print("[INFO] Début du tuning Optuna...")
        study = optuna.create_study(direction='minimize')
        func = lambda trial: objective(trial, X, y, cat_cols)
        study.optimize(func, n_trials=n_trials)
        best_params = study.best_params
        print(f"[INFO] Meilleurs params trouvés : {best_params}")
    else:
        # Params par défaut 
        best_params = {
            'objective': 'regression',
            'metric': ['mae', 'rmse'],
            'boosting_type': 'gbdt',
            'learning_rate': 0.005,
            'num_leaves': 150,
            'max_depth': 12,
            'feature_fraction': 0.6,
            'bagging_fraction': 0.6,
            'bagging_freq': 5,
            'min_data_in_leaf': 15,
            'lambda_l1': 1.0,
            'lambda_l2': 1.0,
            'verbose': -1,
            'seed': 42
        }
        print(f"[INFO] Utilisation des paramètres par défaut : {best_params}")

    # Entraînement Cross-Validation
    print("[INFO] Entraînement avec Cross-Validation (5 folds)...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_list, rmse_list, r2_list, best_iters = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
        print(f"  [Fold {fold}]")
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols)
        valid_data = lgb.Dataset(X_test, label=y_test, categorical_feature=cat_cols, reference=train_data)

        model = lgb.train(
            best_params,
            train_data,
            valid_sets=[train_data, valid_data],
            num_boost_round=5000,
            callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)]
        )

        best_iters.append(model.best_iteration)
        y_pred_log = model.predict(X_test, num_iteration=model.best_iteration)
        y_pred = np.expm1(y_pred_log)
        y_test_orig = np.expm1(y_test)

        mae = mean_absolute_error(y_test_orig, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
        r2 = r2_score(y_test_orig, y_pred)
        print(f"    MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.4f}")

        mae_list.append(mae)
        rmse_list.append(rmse)
        r2_list.append(r2)

    print(f"[INFO] Résultats CV moyens:")
    print(f"  MAE moyenne: {np.mean(mae_list):.2f}")
    print(f"  RMSE moyenne: {np.mean(rmse_list):.2f}")
    print(f"  R² moyenne: {np.mean(r2_list):.4f}")

    # Entraîner modèle final complet
    mean_best_iter = int(np.mean(best_iters))
    print(f"[INFO] Entraînement final complet avec {mean_best_iter} itérations")
    full_data = lgb.Dataset(X, label=y, categorical_feature=cat_cols)
    model = lgb.train(best_params, full_data, num_boost_round=mean_best_iter)

    # Sauvegarde

    save_dir = r"C:\Users\maram\Desktop\stage_2025"
    os.makedirs(save_dir, exist_ok=True)  
    save_prefix = os.path.join(save_dir, 'modele_lightgbm_voiture')

    joblib.dump(model, f'{save_prefix}.pkl')
    joblib.dump(encoder, f'{save_prefix}_encoder.pkl')
    joblib.dump(scaler, f'{save_prefix}_scaler.pkl')

    print(f"[INFO] Modèle sauvegardé dans '{save_prefix}.pkl'")
    print(f"[INFO] Encoder sauvegardé dans '{save_prefix}_encoder.pkl'")
    print(f"[INFO] Scaler sauvegardé dans '{save_prefix}_scaler.pkl'")

    return model, encoder, scaler


if __name__ == "__main__":
    df_path = "C:/Users/maram/Downloads/annonces_brutes_202507021126/annonces_brutes_202507021126.csv"
    print("[START] Test complet du pipeline automatisé")
    model, encoder, scaler = build_and_train_pipeline(df_path=df_path, do_optuna=False, n_trials=10)
    print("[END] Pipeline terminé avec succès")


c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out,

[START] Test complet du pipeline automatisé
[INFO] Chargement des données depuis : C:/Users/maram/Downloads/annonces_brutes_202507021126/annonces_brutes_202507021126.csv
[INFO] Données chargées : 356042 lignes, 22 colonnes
[INFO] Préparation des données...


c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\maram\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out,

[INFO] Données préparées : X=(12431, 39), y=(12431,)
[INFO] Exemple de données préparées (X):


,source,etat,couleur,mise_en_circulation,carburant,boite_de_vitesse,kilometrage,puissance_fiscale,marque,modele,...,full_options,airbags,regulateur_vitesse,abs_esp,importee,nb_portes,suv,berline,coupe,is_premium
26,tayara,avec kilométrage,argent,2010.0,essence,manuelle,2700.0,-0.012379,peugeot,3008,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
39,tayara,nouveau,noir,2024.0,essence,automatique,10000.0,-0.012378,mg,gt,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
56,automobile,inconnu,noir,2021.0,diesel,manuelle,260000.0,-0.012377,hyundai trucks,h350,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0
60,tayara,nouveau,gris,2021.0,essence,automatique,210000.0,-0.012380,kia,picanto,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
69,tayara,nouveau,noir,2011.0,diesel,manuelle,2200.0,-0.012380,volkswagen,golf,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0


[INFO] Utilisation des paramètres par défaut : {'objective': 'regression', 'metric': ['mae', 'rmse'], 'boosting_type': 'gbdt', 'learning_rate': 0.005, 'num_leaves': 150, 'max_depth': 12, 'feature_fraction': 0.6, 'bagging_fraction': 0.6, 'bagging_freq': 5, 'min_data_in_leaf': 15, 'lambda_l1': 1.0, 'lambda_l2': 1.0, 'verbose': -1, 'seed': 42}
[INFO] Entraînement avec Cross-Validation (5 folds)...
  [Fold 1]
Training until validation scores don't improve for 200 rounds
[100]	training's l1: 0.405898	training's rmse: 0.50874	valid_1's l1: 0.410999	valid_1's rmse: 0.515038
[200]	training's l1: 0.304543	training's rmse: 0.401867	valid_1's l1: 0.317212	valid_1's rmse: 0.417632
[300]	training's l1: 0.242564	training's rmse: 0.341144	valid_1's l1: 0.26058	valid_1's rmse: 0.366067
[400]	training's l1: 0.204845	training's rmse: 0.305764	valid_1's l1: 0.227408	valid_1's rmse: 0.33907
[500]	training's l1: 0.181111	training's rmse: 0.283459	valid_1's l1: 0.208431	valid_1's rmse: 0.324459
[600]	traini